In [ ]:
# ========================================
# 1. Import libraries
# ========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore")

# ========================================
# 2. Load dataset
# ========================================

df = pd.read_csv("/kaggle/input/datasets/teamincribo/cyber-security-attacks/cybersecurity_attacks.csv")

print("Original Shape:", df.shape)

# ========================================
# 3. Basic Cleaning
# ========================================

df.columns = df.columns.str.strip()
df = df.drop_duplicates()

# Drop high-cardinality ID columns if present
drop_cols = [c for c in df.columns if "id" in c.lower()]
df = df.drop(columns=drop_cols, errors="ignore")

# ========================================
# 4. Handle Missing Values
# ========================================

for col in df.select_dtypes(include=["object"]):
    df[col] = df[col].fillna(df[col].mode()[0])

for col in df.select_dtypes(include=["int64","float64"]):
    df[col] = df[col].fillna(df[col].median())

# ========================================
# 5. Remove Outliers (IQR)
# ========================================

numeric_cols = df.select_dtypes(include=["int64","float64"]).columns

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("After Cleaning:", df.shape)

# ========================================
# 6. Encode categorical features efficiently
# ========================================

label_encoders = {}

for col in df.select_dtypes(include=["object"]).columns:

    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

    label_encoders[col] = le

# ========================================
# 7. Define Target Variable
# ========================================

target = df.columns[-1]   # usually last column in this dataset

X = df.drop(columns=[target])
y = df[target]

# ========================================
# 8. Train Test Split
# ========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ========================================
# 9. Feature Scaling
# ========================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training Shape:", X_train.shape)